# 05 · Baselines (time-series cross-validation on the training rows)
Baselines of increasing strength: training mean → persistence (PM2.5(t+1) = PM2.5(t)) → single-feature OLS (PM2.5(t) → PM2.5(t+1)) → Random Forest and Gradient Boosting on the raw
features (Set A). **Random Forest and Gradient Boosting are baselines only** and can never be the final model.
All methods use the same five expanding-window folds (`TimeSeriesSplit(5)`, figure S8) and the same eligibility rule; imputer and scaler are fitted inside each training fold.
**Code.** `src/models.py` (`PersistenceRegressor`, `baseline_estimators`, `tune`), `Experiment.stage_baselines`.
The held-out test results of these baselines are in notebook 08 (Table 3).

In [1]:
import io, json, re, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from PIL import Image as PILImage
from IPython.display import Image, Markdown, display
from src import config as C

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 90)

def table(stem, **kw):
    """results/tables/<stem>.csv (written by run_all.py)."""
    return pd.read_csv(C.TABLES / f"{stem}.csv", **kw)

def caption(stem):
    txt = (C.FIGURES / "figure_captions.md").read_text()
    m = re.search(rf"\*\*{re.escape(stem)}\*\* - (.*?)(?:\n\n|\Z)", txt, re.S)
    return m.group(1).strip() if m else ""

def fig(stem, width=1500):
    """Show results/figures/<stem>.png (down-scaled to keep the notebook small) with the caption computed from the data."""
    im = PILImage.open(C.FIGURES / f"{stem}.png")
    if im.width > width:
        im = im.resize((width, round(im.height * width / im.width)), PILImage.LANCZOS)
    buf = io.BytesIO(); im.save(buf, format="PNG", optimize=True)
    display(Image(data=buf.getvalue())); display(Markdown(f"*{caption(stem)}*"))

def log_grep(pattern, maxlen=420):
    """Print the lines of results/logs/run_all.log that match a regular expression."""
    for line in (C.LOGS / "run_all.log").read_text().splitlines():
        if re.search(pattern, line):
            print(line[:maxlen])

## 1. Cross-validated performance (training period, mean ± std over the five folds)

In [2]:
t3 = table("table_3_model_comparison")
base = t3.iloc[:5][["Method", "Features", "CV_MAE", "CV_MAE_std", "CV_RMSE", "CV_RMSE_std", "CV_R2", "CV_R2_std"]]
display(base.round(3))

,Method,Features,CV_MAE,CV_MAE_std,CV_RMSE,CV_RMSE_std,CV_R2,CV_R2_std
0,Training mean,-,62.456,11.291,82.246,19.154,-0.040,0.065
1,Persistence: PM2.5(t+1) = PM2.5(t),PM2.5(t),10.689,1.402,20.228,3.160,0.934,0.017
2,Single-feature OLS: PM2.5(t) -> PM2.5(t+1),PM2.5(t),10.914,1.510,20.100,3.146,0.935,0.016
3,Random Forest (raw features),Set A,11.136,1.956,21.144,5.031,0.930,0.013
4,Gradient Boosting (raw features),Set A,11.764,2.638,22.183,6.227,0.924,0.018


## 2. Hyper-parameters chosen for the tree ensembles (inner search on the same folds)

In [3]:
hp = json.loads((C.MODELS / "selected_hyperparameters.json").read_text())["baselines"]
display(pd.DataFrame(hp).T)

,label,hyperparameters,cv_mae
mean,Training mean,{},62.45638
persistence,Persistence: PM2.5(t+1) = PM2.5(t),{},10.688918
ols_single,Single-feature OLS: PM2.5(t) -> PM2.5(t+1),{},10.914102
rf_raw,Random Forest (raw features),"{'max_features': 0.66, 'min_samples_leaf': 20}",11.135917
hgb_raw,Gradient Boosting (raw features),"{'max_leaf_nodes': 15, 'min_samples_leaf': 100}",11.763875


## 3. Supplementary: the same ensembles on the engineered Set C
Not candidates for the final model; they show whether the engineered features help a flexible model as well (test columns are descriptive).

In [4]:
display(table("table_S10_nonlinear_reference_on_set_c"))

,Method,CV MAE,CV RMSE,Test MAE,Test RMSE,Test R²
0,"Random Forest (Set C, reference only)",10.45 ± 1.72,19.45 ± 4.18,9.53,17.50,0.9572
1,"Gradient Boosting (Set C, reference only)",10.87 ± 2.23,20.52 ± 5.27,9.71,17.94,0.9551
